# General

For more informations, si the documentation *Documentary Strategy*.

# Import & Configs

In [1]:
import json
import pandas as pd

from sentence_transformers import SentenceTransformer
import chromadb

# Upload Data

In [2]:
def get_chunks_df():
    chunks = []
    
    with open("../data/chunks/chunks.jsonl", "r") as f:
    
        for line in f:
    
            chunks.append(json.loads(line))

    return pd.DataFrame(chunks)

In [3]:
chunk_df =  get_chunks_df()

In [4]:
chunk_df.head()

,chunk_id,pmid,chunk_index,title,year,evidence_level,is_review,is_systematic_review,is_meta_analysis,is_guideline,text
0,42277245_0,42277245,0,Mapping radiosensitivity in glioblastoma using...,2026,1,False,False,False,False,Mapping radiosensitivity in glioblastoma using...
1,42277245_1,42277245,1,Mapping radiosensitivity in glioblastoma using...,2026,1,False,False,False,False,While the link between hypoxia and radiosensit...
2,42277245_2,42277245,2,Mapping radiosensitivity in glioblastoma using...,2026,1,False,False,False,False,The model can translate MRE-derived stiffness ...
3,42277245_3,42277245,3,Mapping radiosensitivity in glioblastoma using...,2026,1,False,False,False,False,Our simulations show that biomechanical proper...
4,42277245_4,42277245,4,Mapping radiosensitivity in glioblastoma using...,2026,1,False,False,False,False,The results are compressed vessels with hypo-p...


# Embedding 
## Model

In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Test

In [6]:
sample_embedding = embedding_model.encode(
    chunk_df.iloc[0]["text"]
)

In [7]:
sample_embedding.shape

(384,)

## Chunk Transformation

In [10]:
texts = chunk_df["text"].tolist()

In [12]:
len(texts)

22283

In [13]:
embeddings = embedding_model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True
)

Batches:   0%|          | 0/175 [00:00<?, ?it/s]

In [14]:
embeddings.shape

(22283, 384)

# VectorDB
## Client

In [27]:
client = chromadb.PersistentClient(
    path="../vectorstore/chroma_db"
)

## Collection

Removing old collection:

In [30]:
client.delete_collection("medical_rag")

Create new collection:

In [32]:
collection = client.get_or_create_collection(
    name="medical_rag"
)

In [33]:
collection.count()

0

In [34]:
batch_size = 5000

for start in range(
    0,
    len(chunk_df),
    batch_size
):

    end = start + batch_size

    collection.add(

        ids=chunk_df["chunk_id"].tolist()[start:end],

        documents=chunk_df["text"].tolist()[start:end],

        embeddings=embeddings[start:end].tolist(),

        metadatas=[
            {
                "pmid": str(row["pmid"]),
                "title": str(row["title"]),
                "year": int(row["year"]),

                "evidence_level": int(row["evidence_level"]),

                "is_review": bool(row["is_review"]),
                "is_systematic_review": bool(row["is_systematic_review"]),
                "is_meta_analysis": bool(row["is_meta_analysis"]),
                "is_guideline": bool(row["is_guideline"])
            }

            for _, row in chunk_df.iloc[start:end].iterrows()
        ]
    )

    print(
        f"{min(end, len(chunk_df))}/{len(chunk_df)} inserted"
    )

5000/22283 inserted
10000/22283 inserted
15000/22283 inserted
20000/22283 inserted
22283/22283 inserted


## Retrieval Test

In [38]:
collection.count()

22283

In [64]:
def test_query(query):
    query_embedding = embedding_model.encode(
        query
    )
    
    results = collection.query(
    
        query_embeddings=[query_embedding.tolist()],
    
        n_results=3
    )

    for i, doc in enumerate(results["documents"][0]):
    
        print(f"\nRESULT {i+1}")
        #print(results["metadatas"][0][i]["title"])
        #print(results["metadatas"][0][i]["year"])
        #print(results["metadatas"][0][i]["evidence_level"])
        print(doc[:500])

In [40]:
queries = [
    "MRI diagnosis of glioblastoma",
    
    "brain tumor MRI",

    "glioblastoma prognosis",

    "tumor progression",

    "brain edema",

    "radiotherapy treatment"
]

In [65]:
for query in queries:
    print(f'{query:-^50}')
    test_query(query)
    print("\n\n\n")

----------MRI diagnosis of glioblastoma-----------

RESULT 1
Exploring the Role of Advanced MRI in Understanding Glioblastoma Biology: A Scoping Review. BACKGROUND: Among adult primary brain tumours, glioblastoma (GBM) carries the worst prognosis. Magnetic resonance imaging (MRI) serves to diagnose and guide treatment, despite recognised constraints.

RESULT 2
Neurosurgical and neuro-oncological outcomes of confirmatory brain biopsies in patients with glioblastoma: a real-life monocentric experience. INTRODUCTION: Glioblastoma (GB) is an uncurable tumor with poor prognosis despite resection plus adjuvant cares. When unresectable, even in case of a clear radiological imaging, guidelines require a formal histological diagnosis to confirm the diagnosis of GB.

RESULT 3
Impact on survival of glioblastoma patient's in relation to the imaging of the peri-surgical area: a multi-parametric diffusion MRI, perfusion MRI and [11C]MET PET study. OBJECTIVES: Glioblastoma (GBM) is an aggressive brai

# Dowload Data

*Chroma* already saves all the data in *../vectorstore/chroma_db*. If needed, use the code below to save the data.

In [ ]:
#chunk_df.to_parquet(
#    "../data/processed/chunk_metadata.parquet"
#)